In [1]:
# ============================================================
# PHASE 3 — ADVANCED MODEL
# Cell 1: Environment & Existing Artifact Verification
# ============================================================

import os
import json
import torch
import torchvision

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_DIR = r"C:\Users\rajva\Desktop\image-captioning Project"
NOTEBOOK_DIR = os.path.join(PROJECT_DIR, "notebook")

CHECKPOINT_DIR = os.path.join(NOTEBOOK_DIR, "checkpoints")
FEATURE_DIR = os.path.join(NOTEBOOK_DIR, "cached_features")

print("Project directory:")
print(PROJECT_DIR)

print("\nNotebook directory:")
print(NOTEBOOK_DIR)


# ------------------------------------------------------------
# 2. CPU verification
# ------------------------------------------------------------

device = torch.device("cpu")

print("\nDevice:", device)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)


# ------------------------------------------------------------
# 3. Existing Phase 2 files
# ------------------------------------------------------------

required_files = {
    "Processed captions":
        os.path.join(NOTEBOOK_DIR, "processed_captions.pkl"),

    "Vocabulary":
        os.path.join(NOTEBOOK_DIR, "vocabulary.json"),

    "Train features":
        os.path.join(FEATURE_DIR, "train_features.pt"),

    "Validation features":
        os.path.join(FEATURE_DIR, "val_features.pt"),

    "Test features":
        os.path.join(FEATURE_DIR, "test_features.pt"),

    "Baseline checkpoint":
        os.path.join(CHECKPOINT_DIR, "best_baseline_model.pth"),

    "Baseline metrics":
        os.path.join(NOTEBOOK_DIR, "baseline_metrics.json"),

    "Baseline predictions":
        os.path.join(NOTEBOOK_DIR, "baseline_predictions.json"),
}

print("\n" + "=" * 60)
print("PHASE 2 ARTIFACT CHECK")
print("=" * 60)

all_found = True

for name, path in required_files.items():
    exists = os.path.exists(path)

    status = "FOUND" if exists else "MISSING"

    print(f"{status:8} | {name}")
    print(f"         {path}")

    if not exists:
        all_found = False


# ------------------------------------------------------------
# 4. Load baseline metrics for fixed benchmark
# ------------------------------------------------------------

metrics_path = required_files["Baseline metrics"]

if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        baseline_metrics = json.load(f)

    print("\n" + "=" * 60)
    print("FIXED BASELINE BENCHMARK")
    print("=" * 60)

    for metric, value in baseline_metrics.items():
        print(f"{metric:10}: {value}")


# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)

if all_found:
    print("✓ All required Phase 2 artifacts are available.")
    print("✓ Phase 3 can proceed.")
else:
    print("✗ One or more required files are missing.")
    print("Do NOT continue until the missing files are resolved.")

Project directory:
C:\Users\rajva\Desktop\image-captioning Project

Notebook directory:
C:\Users\rajva\Desktop\image-captioning Project\notebook

Device: cpu
CUDA available: False
PyTorch version: 2.13.0+cpu
Torchvision version: 0.28.0+cpu

PHASE 2 ARTIFACT CHECK
FOUND    | Processed captions
         C:\Users\rajva\Desktop\image-captioning Project\notebook\processed_captions.pkl
FOUND    | Vocabulary
         C:\Users\rajva\Desktop\image-captioning Project\notebook\vocabulary.json
FOUND    | Train features
         C:\Users\rajva\Desktop\image-captioning Project\notebook\cached_features\train_features.pt
FOUND    | Validation features
         C:\Users\rajva\Desktop\image-captioning Project\notebook\cached_features\val_features.pt
FOUND    | Test features
         C:\Users\rajva\Desktop\image-captioning Project\notebook\cached_features\test_features.pt
FOUND    | Baseline checkpoint
         C:\Users\rajva\Desktop\image-captioning Project\notebook\checkpoints\best_baseline_model.pth
F

In [2]:
# ============================================================
# PHASE 3 — CELL 2
# Load and verify pretrained Vision Transformer (ViT-B/16)
# ============================================================

import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

# CPU only
device = torch.device("cpu")

# Load pretrained ViT-B/16
weights = ViT_B_16_Weights.DEFAULT
vit_encoder = vit_b_16(weights=weights)

# Remove classification head to obtain visual representations
vit_encoder.heads = nn.Identity()

# Freeze pretrained parameters
vit_encoder.eval()
for param in vit_encoder.parameters():
    param.requires_grad = False

# Display model configuration
print("Vision Transformer loaded successfully.")

print("\nModel: ViT-B/16")
print("Device:", device)
print("Pretrained weights:", weights)
print("Hidden dimension:", vit_encoder.hidden_dim)
print("Patch size:", vit_encoder.patch_size)
print("Image size:", vit_encoder.image_size)

print(
    "\nAll parameters frozen:",
    all(not p.requires_grad for p in vit_encoder.parameters())
)

print(
    "Classification head removed:",
    isinstance(vit_encoder.heads, nn.Identity)
)

Vision Transformer loaded successfully.

Model: ViT-B/16
Device: cpu
Pretrained weights: ViT_B_16_Weights.IMAGENET1K_V1
Hidden dimension: 768
Patch size: 16
Image size: 224

All parameters frozen: True
Classification head removed: True


In [3]:
# ============================================================
# PHASE 3 — CELL 3
# Verify ViT output shape and feature representation
# ============================================================

import torch

# Create one dummy image: batch size 1, RGB, 224 x 224
dummy_image = torch.randn(1, 3, 224, 224, device=device)

vit_encoder.eval()

with torch.inference_mode():
    dummy_output = vit_encoder(dummy_image)

print("Input shape:", tuple(dummy_image.shape))
print("ViT output shape:", tuple(dummy_output.shape))
print("Output data type:", dummy_output.dtype)
print("Output device:", dummy_output.device)
print("All output values finite:", torch.isfinite(dummy_output).all().item())

# Verify expected global representation
assert dummy_output.shape == (1, 768), "Unexpected ViT output shape!"
assert torch.isfinite(dummy_output).all(), "Output contains NaN or Inf!"

print("\nGlobal feature verification passed.")


Input shape: (1, 3, 224, 224)
ViT output shape: (1, 768)
Output data type: torch.float32
Output device: cpu
All output values finite: True

Global feature verification passed.


In [4]:
# ============================================================
# PHASE 3 — CELL 4
# Extract spatial patch tokens from pretrained ViT-B/16
# ============================================================

import torch

def extract_vit_patch_tokens(vit_model, images):
    """
    Extract spatial patch representations from ViT-B/16.

    Args:
        vit_model: Pretrained torchvision ViT with its
                   classification head replaced by Identity.
        images: Tensor shaped (batch_size, 3, 224, 224).

    Returns:
        Patch tokens shaped (batch_size, 196, 768).
    """
    vit_model.eval()

    with torch.inference_mode():
        # Convert image into patch embeddings
        x = vit_model._process_input(images)

        # Add the class token
        batch_size = x.shape[0]
        cls_token = vit_model.class_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_token, x], dim=1)

        # Apply pretrained positional embeddings and Transformer encoder
        x = vit_model.encoder(x)

        # Remove class token; retain spatial patch tokens
        patch_tokens = x[:, 1:, :]

    return patch_tokens


# Verify using the existing dummy image
patch_tokens = extract_vit_patch_tokens(vit_encoder, dummy_image)

print("Input shape:", tuple(dummy_image.shape))
print("Patch token shape:", tuple(patch_tokens.shape))
print("Number of spatial patches:", patch_tokens.shape[1])
print("Features per patch:", patch_tokens.shape[2])
print("All values finite:", torch.isfinite(patch_tokens).all().item())

assert patch_tokens.shape == (1, 196, 768)
assert torch.isfinite(patch_tokens).all()

print("\nSpatial patch-token verification passed.")

Input shape: (1, 3, 224, 224)
Patch token shape: (1, 196, 768)
Number of spatial patches: 196
Features per patch: 768
All values finite: True

Spatial patch-token verification passed.


In [5]:
# ============================================================
# PHASE 3 — CELL 5
# Verify pretrained ViT image preprocessing
# ============================================================

from torchvision.models import ViT_B_16_Weights

# Official preprocessing for the pretrained weights
vit_weights = ViT_B_16_Weights.DEFAULT
vit_transform = vit_weights.transforms()

print("Preprocessing configuration:")
print(vit_transform)

print("\nExpected image size:", vit_encoder.image_size)
print("Expected channels: 3 (RGB)")
print("Expected patch size:", vit_encoder.patch_size)

print("\nPreprocessing verification information displayed.")

Preprocessing configuration:
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

Expected image size: 224
Expected channels: 3 (RGB)
Expected patch size: 16

Preprocessing verification information displayed.


In [6]:
# ============================================================
# PHASE 3 — CELL 6
# Inspect existing preprocessing variables
# ============================================================

print("Existing notebook variables relevant to image preprocessing:\n")

keywords = [
    "transform",
    "preprocess",
    "dataset",
    "dataloader",
    "image_dir",
    "image_path",
    "train_df",
    "val_df",
    "test_df",
]

for name, value in list(globals().items()):
    if any(keyword in name.lower() for keyword in keywords):
        if not name.startswith("_"):
            print(f"{name}: {type(value).__name__}")

print("\nProject directory exists:", os.path.isdir(PROJECT_DIR))
print("Notebook directory exists:", os.path.isdir(NOTEBOOK_DIR))

Existing notebook variables relevant to image preprocessing:

vit_transform: ImageClassification

Project directory exists: True
Notebook directory exists: True


In [7]:
# ============================================================
# PHASE 3 — CELL 7
# Inspect saved caption dataset and image paths
# ============================================================

import os
import pandas as pd

captions_path = os.path.join(NOTEBOOK_DIR, "processed_captions.pkl")

captions_df = pd.read_pickle(captions_path)

print("Dataset shape:", captions_df.shape)
print("\nColumns:")
print(captions_df.columns.tolist())

print("\nSplit distribution:")
print(captions_df["split"].value_counts())

print("\nUnique images per split:")
print(captions_df.groupby("split")["image"].nunique())

print("\nSample image filenames:")
print(captions_df[["image", "split", "caption"]].head(5).to_string(index=False))

print("\nCaption dataset loaded successfully.")

Dataset shape: (40000, 9)

Columns:
['image', 'caption', 'split', 'clean_caption', 'tokens', 'tokens_with_special', 'token_ids', 'caption_length', 'padded_ids']

Split distribution:
split
train         30000
validation     5000
test           5000
Name: count, dtype: int64

Unique images per split:
split
test          1000
train         6000
validation    1000
Name: image, dtype: int64

Sample image filenames:
                    image split                                                                  caption
1000268201_693b08cb0e.jpg train A child in a pink dress is climbing up a set of stairs in an entry way .
1000268201_693b08cb0e.jpg train                                    A girl going into a wooden building .
1000268201_693b08cb0e.jpg train                         A little girl climbing into a wooden playhouse .
1000268201_693b08cb0e.jpg train                     A little girl climbing the stairs to her playhouse .
1000268201_693b08cb0e.jpg train                A little girl 

In [8]:
# ============================================================
# PHASE 3 — CELL 8
# Locate and verify original dataset images
# ============================================================

RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
IMAGE_DIR = os.path.join(RAW_DIR, "Images")

print("Raw dataset directory exists:", os.path.isdir(RAW_DIR))
print("Image directory:", IMAGE_DIR)
print("Image directory exists:", os.path.isdir(IMAGE_DIR))

if os.path.isdir(IMAGE_DIR):
    image_files = [
        filename for filename in os.listdir(IMAGE_DIR)
        if filename.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("Image files found:", len(image_files))

    print("\nSample file verification:")
    for filename in captions_df["image"].drop_duplicates().head(5):
        path = os.path.join(IMAGE_DIR, filename)
        print(f"{filename}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")
else:
    print("\nImage directory not found. Do not continue until its location is resolved.")

Raw dataset directory exists: True
Image directory: C:\Users\rajva\Desktop\image-captioning Project\data\raw\Images
Image directory exists: True
Image files found: 8091

Sample file verification:
1000268201_693b08cb0e.jpg: FOUND
1001773457_577c3a7d70.jpg: FOUND
1002674143_1b742ab4b8.jpg: FOUND
1003163366_44323f5815.jpg: FOUND
1007129816_e794419615.jpg: FOUND


In [9]:
# ============================================================
# PHASE 3 — CELL 9
# Test preprocessing and patch extraction on a real image
# ============================================================

from PIL import Image
import torch
import os

# Select one existing image from the training split
sample_filename = captions_df.loc[
    captions_df["split"] == "train", "image"
].iloc[0]

sample_path = os.path.join(IMAGE_DIR, sample_filename)

# Load and preprocess the image
with Image.open(sample_path) as img:
    sample_image = img.convert("RGB")
    sample_tensor = vit_transform(sample_image).unsqueeze(0)

# Extract spatial visual tokens
patch_tokens_sample = extract_vit_patch_tokens(
    vit_encoder, sample_tensor.to(device)
)

print("Image filename:", sample_filename)
print("Original image size:", sample_image.size)
print("Preprocessed tensor shape:", tuple(sample_tensor.shape))
print("Patch-token shape:", tuple(patch_tokens_sample.shape))
print("Tensor dtype:", sample_tensor.dtype)
print("All patch values finite:",
      torch.isfinite(patch_tokens_sample).all().item())

assert sample_tensor.shape == (1, 3, 224, 224)
assert patch_tokens_sample.shape == (1, 196, 768)
assert torch.isfinite(patch_tokens_sample).all()

print("\nReal-image preprocessing and extraction verified.")


Image filename: 1000268201_693b08cb0e.jpg
Original image size: (375, 500)
Preprocessed tensor shape: (1, 3, 224, 224)
Patch-token shape: (1, 196, 768)
Tensor dtype: torch.float32
All patch values finite: True

Real-image preprocessing and extraction verified.


In [10]:
# ============================================================
# PHASE 3 — CELL 10
# Prepare a separate ViT feature cache
# ============================================================

import os

VIT_FEATURE_DIR = os.path.join(
    NOTEBOOK_DIR, "vit_cached_features"
)

os.makedirs(VIT_FEATURE_DIR, exist_ok=True)

# Count unique images in each existing split
split_image_counts = (
    captions_df.groupby("split")["image"]
    .nunique()
    .to_dict()
)

print("ViT feature-cache directory:")
print(VIT_FEATURE_DIR)

print("\nDirectory exists:", os.path.isdir(VIT_FEATURE_DIR))

print("\nUnique images per split:")
for split in ["train", "validation", "test"]:
    print(f"{split:12}: {split_image_counts.get(split, 0)}")

print("\nExisting files in ViT cache:")
print(os.listdir(VIT_FEATURE_DIR))

assert split_image_counts["train"] == 6000
assert split_image_counts["validation"] == 1000
assert split_image_counts["test"] == 1000

print("\nViT cache directory and split counts verified.")

ViT feature-cache directory:
C:\Users\rajva\Desktop\image-captioning Project\notebook\vit_cached_features

Directory exists: True

Unique images per split:
train       : 6000
validation  : 1000
test        : 1000

Existing files in ViT cache:
['test_vit_features.pt', 'train_vit_features.pt', 'val_vit_features.pt']

ViT cache directory and split counts verified.


In [11]:
# ============================================================
# PHASE 3 — CELL 11
# Benchmark CPU feature-extraction speed
# ============================================================

import time
from PIL import Image
import torch

# Small benchmark batch
benchmark_filenames = (
    captions_df.loc[captions_df["split"] == "train", "image"]
    .drop_duplicates()
    .head(4)
    .tolist()
)

benchmark_images = []

for filename in benchmark_filenames:
    image_path = os.path.join(IMAGE_DIR, filename)

    with Image.open(image_path) as img:
        benchmark_images.append(
            vit_transform(img.convert("RGB"))
        )

benchmark_batch = torch.stack(benchmark_images).to(device)

# Warm-up pass
with torch.inference_mode():
    _ = extract_vit_patch_tokens(vit_encoder, benchmark_batch)

# Timed pass
start_time = time.perf_counter()

with torch.inference_mode():
    benchmark_features = extract_vit_patch_tokens(
        vit_encoder, benchmark_batch
    )

elapsed = time.perf_counter() - start_time

seconds_per_image = elapsed / len(benchmark_filenames)
estimated_total_minutes = seconds_per_image * 8000 / 60

print("Benchmark batch shape:", tuple(benchmark_batch.shape))
print("Feature shape:", tuple(benchmark_features.shape))
print(f"Images processed: {len(benchmark_filenames)}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Average time per image: {seconds_per_image:.2f} seconds")
print(
    "Rough estimate for 8,000 images:",
    f"{estimated_total_minutes:.1f} minutes"
)
print("\nBenchmark completed.")

Benchmark batch shape: (4, 3, 224, 224)
Feature shape: (4, 196, 768)
Images processed: 4
Elapsed time: 1.38 seconds
Average time per image: 0.35 seconds
Rough estimate for 8,000 images: 46.1 minutes

Benchmark completed.


In [12]:
# ============================================================
# PHASE 3 — CELL 12
# Extract and cache ViT spatial features
# ============================================================

import os
import time
import torch
from PIL import Image

BATCH_SIZE = 4

split_names = {
    "train": "train",
    "validation": "val",
    "test": "test"
}

def extract_and_save_split(split_name, file_prefix):
    filenames = (
        captions_df.loc[captions_df["split"] == split_name, "image"]
        .drop_duplicates()
        .tolist()
    )

    cache_path = os.path.join(
        VIT_FEATURE_DIR, f"{file_prefix}_vit_features.pt"
    )

    features_by_image = {}
    start_time = time.perf_counter()

    print(f"\nProcessing {split_name}: {len(filenames)} images")

    for start_idx in range(0, len(filenames), BATCH_SIZE):
        batch_names = filenames[start_idx:start_idx + BATCH_SIZE]
        batch_tensors = []

        for filename in batch_names:
            image_path = os.path.join(IMAGE_DIR, filename)

            with Image.open(image_path) as img:
                batch_tensors.append(
                    vit_transform(img.convert("RGB"))
                )

        image_batch = torch.stack(batch_tensors).to(device)

        with torch.inference_mode():
            batch_features = extract_vit_patch_tokens(
                vit_encoder, image_batch
            ).cpu()

        for filename, feature in zip(batch_names, batch_features):
            features_by_image[filename] = feature.contiguous()

        processed = min(start_idx + len(batch_names), len(filenames))

        if processed % 400 == 0 or processed == len(filenames):
            elapsed = time.perf_counter() - start_time
            print(
                f"{processed}/{len(filenames)} images | "
                f"{elapsed / 60:.1f} min elapsed"
            )

    torch.save(features_by_image, cache_path)

    elapsed = time.perf_counter() - start_time

    print("Saved:", cache_path)
    print("Images cached:", len(features_by_image))
    print("Example feature shape:",
          tuple(next(iter(features_by_image.values())).shape))
    print(f"Split extraction time: {elapsed / 60:.2f} minutes")

    return cache_path


# Extract each split independently
vit_cache_paths = {}

for split_name, prefix in split_names.items():
    vit_cache_paths[split_name] = extract_and_save_split(
        split_name, prefix
    )

print("\nAll ViT feature extraction jobs completed.")


Processing train: 6000 images
400/6000 images | 2.5 min elapsed
800/6000 images | 5.0 min elapsed
1200/6000 images | 8.0 min elapsed
1600/6000 images | 11.3 min elapsed
2000/6000 images | 15.3 min elapsed
2400/6000 images | 19.4 min elapsed
2800/6000 images | 22.9 min elapsed
3200/6000 images | 24.7 min elapsed
3600/6000 images | 26.5 min elapsed
4000/6000 images | 28.3 min elapsed
4400/6000 images | 30.1 min elapsed
4800/6000 images | 32.5 min elapsed
5200/6000 images | 34.7 min elapsed
5600/6000 images | 36.5 min elapsed
6000/6000 images | 38.3 min elapsed


RuntimeError: [enforce fail at inline_container.cc:745] . open file failed with error code: 1224

In [13]:
import json
from pathlib import Path

vocabulary_path = (
    Path(r"C:\Users\rajva\Desktop\image-captioning Project")
    / "notebook"
    / "vocabulary.json"
)

with open(vocabulary_path, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

VOCAB_SIZE = len(vocab_data)

print("Vocabulary loaded successfully.")
print("Vocabulary size:", VOCAB_SIZE)
print("PAD token:", vocab_data.get("<PAD>"))
print("START token:", vocab_data.get("<START>"))
print("END token:", vocab_data.get("<END>"))

Vocabulary loaded successfully.
Vocabulary size: 12076
PAD token: 0
START token: 1
END token: 2


In [14]:
# ============================================================
# PHASE 3 — CELL 13
# Verify cached ViT features
# ============================================================

import os
import torch

cache_files = {
    "train": os.path.join(
        VIT_FEATURE_DIR, "train_vit_features.pt"
    ),
    "validation": os.path.join(
        VIT_FEATURE_DIR, "val_vit_features.pt"
    ),
    "test": os.path.join(
        VIT_FEATURE_DIR, "test_vit_features.pt"
    ),
}

expected_counts = {
    "train": 6000,
    "validation": 1000,
    "test": 1000,
}

for split, path in cache_files.items():

    assert os.path.isfile(path), f"Missing cache: {path}"

    features = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{split.upper()} SET")
    print("Images:", len(features))
    print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))

    sample_name = next(iter(features))
    sample_feature = features[sample_name]

    print("Sample image:", sample_name)
    print("Feature shape:", tuple(sample_feature.shape))
    print("Data type:", sample_feature.dtype)
    print("All values finite:", torch.isfinite(sample_feature).all().item())

    assert len(features) == expected_counts[split]
    assert sample_feature.shape == (196, 768)
    assert torch.isfinite(sample_feature).all()

print("\nAll cached ViT features verified successfully.")


TRAIN SET
Images: 6000
File size (MB): 3463.82
Sample image: 1000268201_693b08cb0e.jpg
Feature shape: (196, 768)
Data type: torch.float32
All values finite: True

VALIDATION SET
Images: 1000
File size (MB): 577.3
Sample image: 1022454332_6af2c1449a.jpg
Feature shape: (196, 768)
Data type: torch.float32
All values finite: True

TEST SET
Images: 1000
File size (MB): 577.3
Sample image: 1056338697_4f7d7ce270.jpg
Feature shape: (196, 768)
Data type: torch.float32
All values finite: True

All cached ViT features verified successfully.


In [15]:
# ============================================================
# PHASE 3 — CELL 14
# Verify vocabulary and caption token sequences
# ============================================================

import os
import json

vocab_path = os.path.join(NOTEBOOK_DIR, "vocabulary.json")

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

print("Vocabulary JSON type:", type(vocab_data).__name__)

if isinstance(vocab_data, dict):
    print("Vocabulary size:", len(vocab_data))

    # Display special tokens if stored as token -> ID
    for token in ["<PAD>", "<START>", "<END>", "<UNK>"]:
        print(f"{token}: {vocab_data.get(token, 'NOT FOUND')}")

    # Display a sample caption
    sample_row = captions_df.iloc[0]
    print("\nSample image:", sample_row["image"])
    print("Sample caption:", sample_row["caption"])
    print("Token IDs:", sample_row["token_ids"])
    print("Padded IDs:", sample_row["padded_ids"])
else:
    print("Unexpected vocabulary format; inspect the JSON structure.")
    print(str(vocab_data)[:1000])

Vocabulary JSON type: dict
Vocabulary size: 12076
<PAD>: 0
<START>: 1
<END>: 2
<UNK>: 3

Sample image: 1000268201_693b08cb0e.jpg
Sample caption: A child in a pink dress is climbing up a set of stairs in an entry way .
Token IDs: [1, 79, 2023, 5274, 79, 7603, 3232, 5397, 2167, 11322, 79, 9015, 6912, 9976, 5274, 306, 3529, 11661, 2]
Padded IDs: [    1    79  2023  5274    79  7603  3232  5397  2167 11322    79  9015
  6912  9976  5274   306  3529 11661     2     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [16]:
# ============================================================
# PHASE 3 — CELL 15
# Verify caption IDs and image-feature matching
# ============================================================

import os
import torch
import json

# Create ID-to-word mapping for later caption generation
id_to_word = {int(idx): word for word, idx in vocab_data.items()}

# Check special tokens
print("Special-token verification:")
for token in ["<PAD>", "<START>", "<END>", "<UNK>"]:
    print(f"{token}: ID = {vocab_data[token]}")

# Check one image's captions
sample_image_name = captions_df.iloc[0]["image"]
sample_captions = captions_df[
    captions_df["image"] == sample_image_name
]["token_ids"].tolist()

print("\nSample image:", sample_image_name)
print("Number of captions:", len(sample_captions))
print("First caption token count:", len(sample_captions[0]))

# Check feature-cache files without loading the large files
print("\nViT feature-cache files:")
for split, path in vit_cache_paths.items():
    print(f"{split}: {os.path.exists(path)}")

# Estimate feature storage for one image
feature_elements = 196 * 768
print("\nFeature shape per image: (196, 768)")
print("Float32 feature size per image:",
      round(feature_elements * 4 / (1024 ** 2), 2), "MB")

print("\nCell 15 verification completed.")

Special-token verification:
<PAD>: ID = 0
<START>: ID = 1
<END>: ID = 2
<UNK>: ID = 3

Sample image: 1000268201_693b08cb0e.jpg
Number of captions: 5
First caption token count: 19

ViT feature-cache files:

Feature shape per image: (196, 768)
Float32 feature size per image: 0.57 MB

Cell 15 verification completed.


In [17]:
# ============================================================
# PHASE 3 — CELL 16
# Define Transformer Caption Decoder
# ============================================================

import torch
import torch.nn as nn
import math

class TransformerCaptionDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        visual_dim=768,
        embed_dim=256,
        num_heads=8,
        num_layers=3,
        feedforward_dim=1024,
        max_length=38,
        dropout=0.1
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.max_length = max_length

        # Convert token IDs into word embeddings
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        # Project visual patch features into decoder dimension
        self.visual_projection = nn.Linear(
            visual_dim, embed_dim
        )

        # Learnable positional embeddings for caption tokens
        self.position_embedding = nn.Embedding(
            max_length, embed_dim
        )

        # Transformer decoder layer with cross-attention
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        # Predict the next word
        self.output_layer = nn.Linear(
            embed_dim, vocab_size
        )

    def forward(self, visual_features, caption_ids):
        batch_size, caption_length = caption_ids.shape

        # Image patches: (B, 196, 768) -> (B, 196, 256)
        memory = self.visual_projection(visual_features)

        # Word embeddings plus positional information
        positions = torch.arange(
            caption_length, device=caption_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        target = (
            self.embedding(caption_ids) * math.sqrt(self.embed_dim)
            + self.position_embedding(positions)
        )

        # Prevent each word from seeing future words
        causal_mask = torch.triu(
            torch.ones(
                caption_length,
                caption_length,
                device=caption_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Ignore padding tokens in the caption input
        padding_mask = caption_ids.eq(0)

        decoded = self.transformer_decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask
        )

        # Output: (B, caption_length, vocab_size)
        logits = self.output_layer(decoded)

        return logits


print("Transformer decoder class defined successfully.")

Transformer decoder class defined successfully.


In [18]:

VOCAB_SIZE = len(vocab_data)

transformer_decoder = TransformerCaptionDecoder(
    vocab_size=VOCAB_SIZE,
    visual_dim=768,
    embed_dim=256,
    num_heads=8,
    num_layers=3,
    feedforward_dim=1024,
    max_length=38
).to(device)

# Dummy batch: 2 images, 196 patches, 768 features
dummy_visual = torch.randn(2, 196, 768, device=device)

# Dummy captions: 2 sequences, each 10 tokens
dummy_captions = torch.randint(
    low=4,
    high=VOCAB_SIZE,
    size=(2, 10),
    device=device
)
dummy_captions[:, 0] = 1  # <START>

transformer_decoder.eval()

with torch.inference_mode():
    dummy_logits = transformer_decoder(
        dummy_visual, dummy_captions
    )

total_params = sum(
    p.numel() for p in transformer_decoder.parameters()
)
trainable_params = sum(
    p.numel() for p in transformer_decoder.parameters()
    if p.requires_grad
)

print("Visual input shape:", tuple(dummy_visual.shape))
print("Caption input shape:", tuple(dummy_captions.shape))
print("Output logits shape:", tuple(dummy_logits.shape))
print("Expected output shape:", (2, 10, VOCAB_SIZE))
print(f"Total decoder parameters: {total_params:,}")
print(f"Trainable decoder parameters: {trainable_params:,}")
print("All output values finite:", torch.isfinite(dummy_logits).all().item())

Visual input shape: (2, 196, 768)
Caption input shape: (2, 10)
Output logits shape: (2, 10, 12076)
Expected output shape: (2, 10, 12076)
Total decoder parameters: 9,561,900
Trainable decoder parameters: 9,561,900
All output values finite: True
